# Fine-grained MCTS → router training data

This notebook walks through the **same functions** used when building `fine_routing_*` datasets (`data_prep/build_fine_routing_dataset.py`) and when **loading** them for training (`experiments/sweep_fine_routing.py`).

## Setup

- From repo root: `pip install -r requirements.txt` (you already need this for the project).
- To run this file as a notebook: `pip install jupyter` or `pip install jupyterlab`, then e.g. `jupyter lab notebooks/fine_grained_mcts_data_inspection.ipynb` from the repo root.
- Cells that call `prepare_arc_data` may download Hugging Face datasets on first use (network).
- Cells that load a real `*_pivot_residuals.pt` file expect PyTorch with `weights_only=True` support (recent `torch`).

Edit the paths in the **configuration** cell to match your machine.

## 1. Repository root and imports

In [1]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

# Repo root = parent of this notebook's folder
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /home/janerik/flexible-test-time-program-selection


## 2. Paths and `FineRoutingConfig`

Adjust `RESULTS_DIR` (MCTS snapshots / `benchmark_mcts_*_snapshot.json`) and `DATA_DIR` (output of `build_fine_routing_dataset.py` with `--use_mcts`).

In [2]:
from routers.fine_routing_config import FineRoutingConfig

# Directory with benchmark_mcts_<bench>_*_snapshot.json (or final_val_*)
RESULTS_DIR = os.environ.get(
    "FINE_ROUTING_RESULTS_DIR",
    str(REPO_ROOT / "predictions"),  # change if your snapshots live elsewhere
)

# Built dataset directory (contains config.json, <bench>.jsonl, <bench>_pivot_residuals.pt)
DATA_DIR = os.environ.get(
    "FINE_ROUTING_DATA_DIR",
    str(REPO_ROOT / "fine_routing_data"),
)

BENCHMARK = os.environ.get("FINE_ROUTING_BENCHMARK", "arc_easy")
MODEL_NAME = os.environ.get("FINE_ROUTING_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")

cfg = FineRoutingConfig(model_name=MODEL_NAME, results_dir=RESULTS_DIR)
cfg.benchmarks = [BENCHMARK]
cfg.use_mcts = True

print("RESULTS_DIR:", RESULTS_DIR)
print("DATA_DIR:", DATA_DIR)
print("BENCHMARK:", BENCHMARK)
json.dumps(asdict(cfg), indent=2)[:2000]  # truncated; full dict is large

RESULTS_DIR: /home/janerik/flexible-test-time-program-selection/predictions
DATA_DIR: /home/janerik/flexible-test-time-program-selection/fine_routing_data
BENCHMARK: arc_easy


'{\n  "model_name": "Qwen/Qwen2.5-0.5B-Instruct",\n  "results_dir": "/home/janerik/flexible-test-time-program-selection/predictions",\n  "benchmarks": [\n    "arc_easy"\n  ],\n  "gpu_rank": 0,\n  "pivot_layer": 16,\n  "pivot_layers_multi": [\n    12,\n    14,\n    16\n  ],\n  "use_multi_layer_pivot": false,\n  "include_anchor_confidence": false,\n  "editable_start": 17,\n  "max_local_edits": 2,\n  "swap_radius": 2,\n  "enumerate_deviations": true,\n  "use_mcts": true,\n  "mcts_num_simulations": 64,\n  "mcts_exploration_constant": 1.8,\n  "mcts_pw_C": 1.0,\n  "mcts_pw_alpha": 0.5,\n  "num_local_search_sims": 100,\n  "data_split": "train",\n  "output_dir": "fine_routing_data",\n  "delta_clip": 1.0,\n  "target_beta": 5.0,\n  "gate_tau": 0.0,\n  "use_continuous_scoring": false,\n  "continuous_delta_clip": 5.0,\n  "continuous_target_beta": 2.0,\n  "continuous_gate_tau": 0.1,\n  "mcts_dual_seed": false,\n  "gate_threshold_gamma": 0.8,\n  "gate_w1": 5.0,\n  "gate_w0": 1.0,\n  "gate_hidden_dim

## 3. Anchor sequence from MCTS results

Same helper as dataset CLI: `training.train_benchmark_router.load_optimal_sequences_from_results`.

In [3]:
from training.train_benchmark_router import load_optimal_sequences_from_results

anchor_seqs = load_optimal_sequences_from_results(
    RESULTS_DIR,
    [BENCHMARK],
    model_name=MODEL_NAME,
)
anchor_seq = anchor_seqs.get(BENCHMARK)
print("keys found:", list(anchor_seqs.keys()))
print("anchor_seq (first 12)", anchor_seq[:12] if anchor_seq else None, "... len", len(anchor_seq) if anchor_seq else None)

if anchor_seq is None:
    num_layers = 24  # Qwen2.5-0.5B typical; override with your model
    anchor_seq = list(range(num_layers))
    print("Using fallback identity anchor:", anchor_seq[:8], "...")

keys found: []
anchor_seq (first 12) None ... len None
Using fallback identity anchor: [0, 1, 2, 3, 4, 5, 6, 7] ...


## 4. Benchmark samples (`prepare_arc_data`)

Same as `build_fine_routing_dataset.main`: questions are loaded with `prepare_arc_data` (HF datasets).

In [4]:
from core.flexible_models import get_is_instruct
from core.permutation_mcts import prepare_arc_data

is_instruct = get_is_instruct(MODEL_NAME)
samples = prepare_arc_data(BENCHMARK, is_instruct=is_instruct, split=cfg.data_split)
samples = samples[:3]  # small slice for inspection

print("n_samples (truncated):", len(samples))
s0 = samples[0]
print("sample keys:", sorted(s0.keys()))
print("input snippet:", s0["input"][:200].replace("\n", " "), "...")
print("correct:", repr(s0.get("correct")), "max_new_tokens:", s0.get("max_new_tokens"))

2026-04-28 08:11:02,381 INFO HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-28 08:11:02,394 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/allenai/ai2_arc/210d026faf9955653af8916fad021475a3f00453/README.md "HTTP/1.1 200 OK"
2026-04-28 08:11:02,509 INFO HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 404 Not Found"


2026-04-28 08:11:02,510 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-28 08:11:02,871 INFO HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/allenai/ai2_arc/allenai/ai2_arc.py "HTTP/1.1 404 Not Found"
2026-04-28 08:11:02,995 INFO HTTP Request: GET https://huggingface.co/api/datasets/allenai/ai2_arc/revision/210d026faf9955653af8916fad021475a3f00453 "HTTP/1.1 200 OK"
2026-04-28 08:11:03,111 INFO HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-04-28 08:11:03,283 INFO HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=allenai/ai2_arc "HTTP/1.1 200 OK"
2026-04-28 08:11:03,405 INFO HTTP Request: GET https://huggingface.co/api/datasets/allenai/ai2_arc/tree/210d026faf9955653af8916fad021475a3f00453/ARC-Challenge?recursiv

Preparing samples: 100%|████████████| 2251/2251 [00:00<00:00, 21410.77it/s]

n_samples (truncated): 3
sample keys: ['correct', 'input']
input snippet: Question: Which factor will most likely cause a person to develop a fever? A. a leg muscle relaxing after exercise B. a bacterial population in the bloodstream C. several viral particles on the skin D ...
correct: 'B' max_new_tokens: None


## 5. Router target from deltas (`compute_router_target`)

Used for both exhaustive and MCTS modes: softmax over clipped deltas.

In [5]:
from data_prep.build_fine_routing_dataset import compute_router_target

deltas_demo = [0.0, 0.2, -0.1, 0.5]
pi = compute_router_target(deltas_demo, beta=cfg.target_beta, clip_val=cfg.delta_clip)
print("deltas:", deltas_demo)
print("pi (sum=1):", pi, "sum", sum(pi))

deltas: [0.0, 0.2, -0.1, 0.5]
pi (sum=1): [0.06057923520697361, 0.16467143424506278, 0.03674316349497249, 0.7380061670529912] sum 1.0


## 6. Per-question MCTS (`per_question_mcts`)

Core search routine from `core.benchmark_mcts`. Here we use a **synthetic** `grade_fn` so you can run without loading a GPU model. In production, `grade_fn` wraps `grade_sample` or `grade_sample_continuous` on the real wrapper.

In [ ]:
from core.benchmark_mcts import per_question_mcts, seq_to_layers

num_layers = len(anchor_seq)


def toy_grade_fn(seq: list[int]) -> float:
    layers = seq_to_layers(seq)
    if not layers:
        return 0.0
    # deterministic toy reward: spread of late-layer ids
    tail = layers[-4:]
    return float(max(tail) - min(tail)) / max(num_layers, 1)


mcts_result = per_question_mcts(
    anchor_seq=list(anchor_seq),
    grade_fn=toy_grade_fn,
    num_simulations=32,
    num_layers=num_layers,
    radius=cfg.swap_radius,
    max_swaps=cfg.max_local_edits,
    editable_start=cfg.editable_start,
    exploration_constant=cfg.mcts_exploration_constant,
    pw_C=cfg.mcts_pw_C,
    pw_alpha=cfg.mcts_pw_alpha,
)

print("top-level keys:", sorted(mcts_result.keys()))
print("anchor_score:", mcts_result["anchor_score"])
print("best_delta:", mcts_result["best_delta"], "num_explored:", mcts_result["num_explored"])
print("best_seq (first 12):", mcts_result["best_seq"][:12])

## 7. Jsonl record shape (mirrors `build_dataset_mcts_for_benchmark`)

Below we assemble one **logical row** like the writer in `build_dataset_mcts_for_benchmark`: `explored` becomes a **list of dicts** on disk (`explored_list`), not the raw `cache` dict.

In [ ]:
import hashlib


def sample_hash(sample: dict) -> str:
    text = sample["input"] + str(sample.get("correct", ""))
    return hashlib.md5(text.encode()).hexdigest()


explored = mcts_result["explored"]
seq_keys = list(explored.keys())
anchor_score = mcts_result["anchor_score"]
deltas = [explored[k] - anchor_score for k in seq_keys]
router_target = compute_router_target(
    deltas, beta=cfg.target_beta, clip_val=cfg.delta_clip
)

explored_list = []
for k in seq_keys:
    explored_list.append(
        {
            "seq": list(k),
            "score": explored[k],
            "delta": explored[k] - anchor_score,
            "score_binary": explored[k],  # real pipeline fills binary/continuous via grade_sample_both
            "delta_binary": explored[k] - anchor_score,
            "score_continuous": explored[k],
            "delta_continuous": explored[k] - anchor_score,
        }
    )

rec = {
    "benchmark_id": BENCHMARK,
    "question_id": 0,
    "question_hash": sample_hash(samples[0]),
    "anchor_sequence": list(anchor_seq),
    "anchor_score": anchor_score,
    "anchor_score_binary": toy_grade_fn(list(anchor_seq)),
    "anchor_score_continuous": toy_grade_fn(list(anchor_seq)),
    "pivot_layer_index": cfg.pivot_layer,
    "search_mode": "mcts",
    "scoring_mode": "both",
    "primary_scoring": "binary",
    "gate_label": int(mcts_result["best_delta"] > cfg.gate_tau),
    "best_seq": mcts_result["best_seq"],
    "best_score": mcts_result["best_score"],
    "best_score_binary": mcts_result["best_score"],
    "best_score_continuous": mcts_result["best_score"],
    "best_delta": mcts_result["best_delta"],
    "best_delta_binary": mcts_result["best_delta"],
    "best_delta_continuous": mcts_result["best_delta"],
    "num_explored": mcts_result["num_explored"],
    "explored": explored_list,
    "router_target": router_target,
    "router_target_binary": router_target,
    "router_target_continuous": router_target,
}

print("record keys:", sorted(rec.keys()))
print("len(router_target)", len(rec["router_target"]), "vs len(explored)", len(rec["explored"]))
print("router_target sums to", sum(rec["router_target"]))
print(json.dumps({k: rec[k] for k in rec if k != "explored"}, indent=2)[:1500])

## 8. Training loader: catalog + soft targets

`load_bench_data_mcts` builds the **global sequence catalog** from all `explored` rows, then maps each question's `router_target` into a fixed `[num_classes]` vector.

In [ ]:
import torch

from experiments.sweep_fine_routing import (
    build_mcts_router_targets,
    build_mcts_sequence_catalog,
    build_reduced_catalog,
    load_bench_data_mcts,
)

records_demo = [rec]
catalog_full, seq_to_idx_full = build_mcts_sequence_catalog(records_demo, list(anchor_seq))
catalog_reduced, seq_to_idx_reduced = build_reduced_catalog(records_demo, list(anchor_seq))
n_full = len(catalog_full)
router_targets = build_mcts_router_targets(records_demo, seq_to_idx_full, n_full)

print("|catalog_full|:", n_full, "|catalog_reduced|:", len(catalog_reduced))
print("one target shape:", router_targets[0].shape, "sum:", float(router_targets[0].sum()))

## 9. Load real on-disk dataset (optional)

Runs only if `DATA_DIR/<benchmark>.jsonl` and `_pivot_residuals.pt` exist.

In [ ]:
pt_path = Path(DATA_DIR) / f"{BENCHMARK}_pivot_residuals.pt"
jsonl_path = Path(DATA_DIR) / f"{BENCHMARK}.jsonl"

if pt_path.is_file() and jsonl_path.is_file():
    (
        residuals,
        gate_labels,
        router_targets,
        catalog_full,
        seq_to_idx_full,
        catalog_reduced,
        seq_to_idx_reduced,
        records,
    ) = load_bench_data_mcts(str(DATA_DIR), BENCHMARK, list(anchor_seq))

    print("residuals:", tuple(residuals.shape), residuals.dtype)
    print("n_records:", len(records), "gate positives:", sum(gate_labels))
    print("|catalog_full|:", len(catalog_full), "|catalog_reduced|:", len(catalog_reduced))
    r0 = records[0]
    assert r0.get("search_mode") == "mcts", "Expected MCTS rows for this notebook path"
    print("first record keys:", sorted(r0.keys()))
    print("first explored entry keys:", sorted(r0["explored"][0].keys()))
else:
    print("Skip: missing files")
    print(" ", pt_path)
    print(" ", jsonl_path)

## 10. Sanity: on-disk `router_target` aligns with `explored`

For MCTS jsonl, `len(record["router_target"])` should match `len(record["explored"])`, and probabilities should sum to 1.

In [ ]:
if jsonl_path.is_file():
    with open(jsonl_path) as f:
        line = f.readline()
    row = json.loads(line)
    le, lr = len(row["explored"]), len(row["router_target"])
    print("explored vs router_target len:", le, lr, "OK" if le == lr else "MISMATCH")
    print("router_target sum:", sum(row["router_target"]))
    print("search_mode:", row.get("search_mode"), "primary_scoring:", row.get("primary_scoring"))
else:
    print("No jsonl at", jsonl_path)